# REVE frozen Head A-eng — engagement corpus

Private GPU kernel — **select T4 in UI** (avoid P100; current Kaggle PyTorch needs sm_70+).
Offline REVE from `muse-eeg-heads-cache`. HF_TOKEN UserSecrets = fallback only.

- Corpus: `muse-eeg-heads-aeng` / engagement_a_eng (~133 persons, muse4, 93/20/20)
- Labels: low_engagement / high_engagement
- Frozen REVE-base + HeadAEngLinear — **no backbone fine-tune**
- If P100 assigned: auto-fallback to **CPU** encode (slow but correct)


In [ ]:
import os, sys, json, gc, time, shutil, subprocess, zipfile
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
INPUT_ROOT = Path('/kaggle/input')
OUT = WORKING / 'head_a_eng_train_reve'
OUT.mkdir(parents=True, exist_ok=True)
EMB = OUT / 'emb_cache'
EMB.mkdir(exist_ok=True)
LOG = []

def log(msg):
    line = f"[{datetime.now(timezone.utc).isoformat()}] {msg}"
    print(line, flush=True)
    LOG.append(line)

def find_dir(names, markers):
    if not INPUT_ROOT.exists():
        return None
    for name in names:
        d = INPUT_ROOT / name
        if d.exists():
            for g in markers:
                if list(d.glob(g)) or list(d.rglob(g.split('/')[-1])):
                    return d
    for g in markers:
        hits = list(INPUT_ROOT.rglob(g.split('/')[-1]))
        if hits:
            # climb until marker parent makes sense
            return hits[0].parents[min(2, len(hits[0].parents)-1)]
    return None

def unzip_nested(root: Path):
    if root is None or not root.exists():
        return
    for zpath in list(root.glob('*.zip')) + list(root.rglob('*.zip')):
        dest = zpath.with_suffix('')
        if dest.exists():
            continue
        try:
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(dest)
            log(f'unzipped {zpath.name} -> {dest}')
        except Exception as e:
            log(f'unzip skip {zpath}: {e}')

# deps
pkgs = ['transformers>=4.40', 'einops', 'accelerate', 'safetensors', 'huggingface_hub']
uv = shutil.which('uv')
if uv:
    subprocess.check_call([uv, 'pip', 'install', '--system', '-q', *pkgs])
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
log(f'torch={torch.__version__} cuda={torch.cuda.is_available()} gpu={gpu_name}')
force_cpu = False
if torch.cuda.is_available() and ('P100' in gpu_name):
    log('WARNING: P100 detected — PyTorch on Kaggle often lacks sm_60; forcing CPU fallback. Re-run with T4 in UI.')
    force_cpu = True
device = torch.device('cpu' if force_cpu or not torch.cuda.is_available() else 'cuda')
log(f'device={device}')

src_dir = find_dir(('muse-eeg-heads-src',), ('reve_encoder.py',))
aeng_dir = find_dir(('muse-eeg-heads-aeng',), ('splits.json', '*_aeng_windows.npz'))
cache_dir = find_dir(('muse-eeg-heads-cache',), ('model.safetensors', 'MANIFEST.md'))
log(f'src_dir={src_dir}')
log(f'aeng_dir={aeng_dir}')
log(f'cache_dir={cache_dir}')
unzip_nested(src_dir)
unzip_nested(aeng_dir)
# re-find after unzip
src_dir = find_dir(('muse-eeg-heads-src',), ('reve_encoder.py',))
aeng_dir = find_dir(('muse-eeg-heads-aeng',), ('splits.json',))
if src_dir is None:
    raise FileNotFoundError('muse-eeg-heads-src / reve_encoder.py not found under /kaggle/input')
if aeng_dir is None:
    raise FileNotFoundError('muse-eeg-heads-aeng not found')

# locate windows + splits
WIN = None
SPLITS = None
for cand in [aeng_dir / 'engagement_a_eng' / 'windows', aeng_dir / 'windows']:
    if cand.exists() and list(cand.glob('*_aeng_windows.npz')):
        WIN = cand
        break
if WIN is None:
    hits = list(aeng_dir.rglob('*_aeng_windows.npz'))
    if hits:
        WIN = hits[0].parent
for cand in [aeng_dir / 'engagement_a_eng' / 'splits' / 'splits.json', aeng_dir / 'splits' / 'splits.json']:
    if cand.exists():
        SPLITS = cand
        break
if SPLITS is None:
    hits = list(aeng_dir.rglob('splits.json'))
    if hits:
        SPLITS = hits[0]
nwin = len(list(WIN.glob('*_windows.npz'))) if WIN else 0
log(f'WIN={WIN} n={nwin}')
log(f'SPLITS={SPLITS}')
assert WIN and SPLITS

# sys.path: src root and any nested src/
sys.path.insert(0, str(src_dir))
for p in src_dir.rglob('reve_encoder.py'):
    sys.path.insert(0, str(p.parent))
    break
for p in src_dir.rglob('head_a_eng.py'):
    sys.path.insert(0, str(p.parent.parent))  # .../heads/../
    break

from reve_encoder import FrozenREVEEncoder
from heads.head_a_eng import HEAD_A_ENG_LABELS, HeadAEngLinear
from heads.base import class_weights_from_y, undersample_balanced
from metrics import confusion_matrix, per_class_report
log('imports OK')


In [ ]:
# HF_TOKEN fallback only
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
        os.environ['HUGGING_FACE_HUB_TOKEN'] = tok
        os.environ['HUGGINGFACE_HUB_TOKEN'] = tok
        log('HF_TOKEN loaded from secrets (fallback only)')
except Exception as e:
    log(f'no HF_TOKEN secret (ok if cache present): {type(e).__name__}: {e}')

SEED = 42
TARGET_SR = 256.0
BATCH_ENC = 8 if device.type == 'cpu' else 16
BATCH_HEAD = 256
EPOCHS = 20
LR = 1e-3
PATIENCE = 5
MAX_TRAIN = 40000
CHANCE = 0.5
SHIP_MIN = 0.55
CLEANER = {'ds007262'}

splits = json.loads(SPLITS.read_text())
split_map = {}
for sp, persons in splits['splits'].items():
    for pid in persons:
        split_map[pid] = sp
persons = splits['persons']
packs = []
for pid, info in persons.items():
    sp = split_map.get(pid)
    if sp is None:
        continue
    for pk in info['packs']:
        packs.append((pk, pid, sp))
packs.sort()
log(f"persons={splits['n_unique_persons']} packs={len(packs)}")

extra_roots = []
if cache_dir is not None:
    extra_roots.append(cache_dir / 'models')
    extra_roots.append(cache_dir)
enc = FrozenREVEEncoder(source_sr=TARGET_SR, cache_roots=extra_roots, prefer_local=True).to(device)
enc.eval()
for p in enc.parameters():
    p.requires_grad_(False)
log(f"emb_dim={enc.emb_dim} model_from_local={getattr(enc,'model_from_local',None)} model_id={getattr(enc,'model_id',None)}")


In [ ]:
@torch.no_grad()
def encode_batch(X):
    chunks = []
    for i in range(0, len(X), BATCH_ENC):
        xb = torch.from_numpy(X[i:i+BATCH_ENC]).to(device)
        chunks.append(enc(xb).cpu().numpy().astype(np.float32))
        if device.type == 'cuda' and (i // BATCH_ENC) % 50 == 0:
            torch.cuda.empty_cache()
    return np.concatenate(chunks, 0) if chunks else np.zeros((0, enc.emb_dim), np.float32)

def remap_y(y_raw, label_names):
    table = {n:i for i,n in enumerate(HEAD_A_ENG_LABELS)}
    y = np.full(len(y_raw), -1, np.int64)
    for oi, name in enumerate(label_names):
        if str(name) in table:
            y[y_raw == oi] = table[str(name)]
    return y

t0 = time.time()
meta = {'packs': []}
for i, (pk, pid, sp) in enumerate(packs):
    cache_path = EMB / f'{pk}_emb.npz'
    npz_path = WIN / f'{pk}_aeng_windows.npz'
    man_path = WIN / f'{pk}_aeng_manifest.json'
    source = pk.split('_')[0]
    device_class = 'unknown'
    if man_path.exists():
        man = json.loads(man_path.read_text())
        source = man.get('source', source)
        device_class = man.get('device_class', device_class)
    if cache_path.exists():
        z = np.load(cache_path, allow_pickle=True)
        meta['packs'].append({'pack_key': pk, 'n': int(z['emb'].shape[0]), 'cached': True, 'source': source, 'split': sp})
        continue
    if not npz_path.exists():
        log(f'SKIP missing {pk}')
        continue
    data = np.load(npz_path, allow_pickle=True)
    X = data['X'].astype(np.float32)
    y = remap_y(data['y'].astype(np.int64), [str(x) for x in data['label_names'].tolist()])
    keep = y >= 0
    X, y = X[keep], y[keep]
    emb = encode_batch(X)
    np.savez_compressed(cache_path, emb=emb.astype(np.float32), y=y.astype(np.int64),
                        unique_person_id=np.asarray(pid), split=np.asarray(sp),
                        pack_key=np.asarray(pk), source=np.asarray(source),
                        device_class=np.asarray(device_class))
    meta['packs'].append({'pack_key': pk, 'n': int(len(y)), 'cached': False, 'source': source, 'split': sp})
    if (i+1) % 10 == 0 or i < 3:
        rate = (i+1) / max(time.time()-t0, 1e-6)
        log(f'encoded {i+1}/{len(packs)} {pk} split={sp} n={len(y)} elapsed={time.time()-t0:.0f}s rate={rate:.2f} packs/s')
    del data, X, emb
    gc.collect()
(EMB / 'cache_meta.json').write_text(json.dumps(meta, indent=2))
log(f'encode done in {time.time()-t0:.0f}s')


In [ ]:
def load_pooled(split, source_filter=None):
    embs, ys = [], []
    detail = {'packs': [], 'counts': Counter(), 'n_persons': 0, 'sources': Counter(), 'device_classes': Counter()}
    persons_set = set()
    for path in sorted(EMB.glob('*_emb.npz')):
        z = np.load(path, allow_pickle=True)
        if str(z['split']) != split:
            continue
        src = str(z['source']) if 'source' in z.files else 'unknown'
        if source_filter is not None and src not in source_filter:
            continue
        emb = z['emb'].astype(np.float32)
        y = z['y'].astype(np.int64)
        m = y >= 0
        emb, y = emb[m], y[m]
        if len(y) == 0:
            continue
        embs.append(emb); ys.append(y)
        pid = str(z['unique_person_id']); persons_set.add(pid)
        dc = str(z['device_class']) if 'device_class' in z.files else 'unknown'
        c = Counter({HEAD_A_ENG_LABELS[j]: int((y==j).sum()) for j in range(2)})
        detail['packs'].append({'pack_key': str(z['pack_key']), 'unique_person_id': pid, 'source': src, 'n': int(len(y)), 'counts': dict(c)})
        detail['counts'].update(c); detail['sources'][src] += int(len(y)); detail['device_classes'][dc] += int(len(y))
    if not embs:
        raise RuntimeError(f'empty split {split}')
    detail['counts']=dict(detail['counts']); detail['sources']=dict(detail['sources'])
    detail['device_classes']=dict(detail['device_classes']); detail['n_persons']=len(persons_set)
    return np.concatenate(embs), np.concatenate(ys), detail

def eval_head(head, emb, y):
    head.eval(); preds=[]
    with torch.no_grad():
        for i in range(0, len(emb), BATCH_HEAD):
            logits = head(torch.from_numpy(emb[i:i+BATCH_HEAD]).to(device))
            preds.extend(logits.argmax(-1).cpu().numpy().tolist())
    pred = np.asarray(preds, np.int64)
    report = per_class_report(y.tolist(), pred.tolist(), list(HEAD_A_ENG_LABELS))
    return {
        'n': int(len(y)),
        'accuracy': float((pred==y).mean()) if len(y) else 0.0,
        'macro_f1': float(report['macro_f1']['f1']),
        'per_class': {k:v for k,v in report.items() if k!='macro_f1'},
        'confusion_matrix': confusion_matrix(y.tolist(), pred.tolist(), list(HEAD_A_ENG_LABELS)).tolist(),
        'pred_counts': {HEAD_A_ENG_LABELS[i]: int((pred==i).sum()) for i in range(2)},
        'true_counts': {HEAD_A_ENG_LABELS[i]: int((y==i).sum()) for i in range(2)},
        'delta_vs_chance': float(report['macro_f1']['f1']) - CHANCE,
    }

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
emb_tr, y_tr, det_tr = load_pooled('train')
emb_va, y_va, det_va = load_pooled('val')
emb_te, y_te, det_te = load_pooled('test')
log(f'train={len(y_tr)} {det_tr["sources"]} val={len(y_va)} test={len(y_te)}')
emb_fit, y_fit = undersample_balanced(emb_tr, y_tr, rng=rng)
if len(y_fit) > MAX_TRAIN:
    per = MAX_TRAIN // 2
    picks = [rng.choice(np.where(y_fit==c)[0], size=min(per, int((y_fit==c).sum())), replace=False) for c in (0,1)]
    sel = np.concatenate(picks); rng.shuffle(sel)
    emb_fit, y_fit = emb_fit[sel], y_fit[sel]
log(f'fit={len(y_fit)} {Counter(y_fit.tolist())}')

head = HeadAEngLinear(in_dim=int(emb_fit.shape[-1])).to(device)
w = class_weights_from_y(y_fit, n_classes=2)
crit = nn.CrossEntropyLoss(weight=w.to(device))
opt = torch.optim.Adam(head.parameters(), lr=LR)
loader = DataLoader(TensorDataset(torch.from_numpy(emb_fit), torch.from_numpy(y_fit)), batch_size=BATCH_HEAD, shuffle=True)
history=[]; best=-1.0; best_state=None; stale=0
for ep in range(EPOCHS):
    head.train(); total=n=0
    for xb,yb in loader:
        xb,yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss=crit(head(xb), yb); loss.backward(); opt.step()
        total += float(loss.item())*len(yb); n += len(yb)
    tr = eval_head(head, emb_fit, y_fit); va = eval_head(head, emb_va, y_va)
    row = {'epoch': ep+1, 'loss': total/max(n,1), 'train_macro_f1': tr['macro_f1'], 'val_macro_f1': va['macro_f1'], 'val_acc': va['accuracy']}
    history.append(row); log(str(row))
    if va['macro_f1'] > best + 1e-4:
        best = va['macro_f1']; best_state = {k:v.detach().cpu().clone() for k,v in head.state_dict().items()}; stale=0
    else:
        stale += 1
        if stale >= PATIENCE:
            log(f'early stop {ep+1}'); break
if best_state: head.load_state_dict(best_state)
train_m = eval_head(head, emb_fit, y_fit)
val_m = eval_head(head, emb_va, y_va)
test_m = eval_head(head, emb_te, y_te)
log(f"TEST n={test_m['n']} acc={test_m['accuracy']:.3f} macro_f1={test_m['macro_f1']:.3f} delta={test_m['delta_vs_chance']:+.3f}")

by_source = {}
for src in sorted({p['source'] for p in det_te['packs']}):
    try:
        e_s, y_s, _ = load_pooled('test', source_filter={src})
        by_source[src] = eval_head(head, e_s, y_s)
        log(f"  source {src}: n={by_source[src]['n']} macro_f1={by_source[src]['macro_f1']:.3f}")
    except Exception as e:
        log(f'src fail {src}: {e}')

reasons=[]; ok=True
if test_m['macro_f1'] < SHIP_MIN:
    ok=False; reasons.append(f"test F1 {test_m['macro_f1']:.3f} < {SHIP_MIN}")
for name in HEAD_A_ENG_LABELS:
    if test_m['per_class'][name]['f1'] < 0.45:
        ok=False; reasons.append(f'{name} F1 low')
    if test_m['pred_counts'].get(name,0)==0:
        ok=False; reasons.append(f'never predicts {name}')
cleaner_ok=False
for src in CLEANER:
    if src in by_source and by_source[src]['n']>=50 and by_source[src]['macro_f1']>=SHIP_MIN:
        cleaner_ok=True
    elif src in by_source:
        reasons.append(f"cleaner {src} F1 {by_source[src]['macro_f1']:.3f}")
confound = 'Residual order confounds ds007169/eegmat/STEW; domain mix pro vs STEW hobbyist; muse4 proxy.'
if not ok:
    ship=False; ship_reason='; '.join(reasons)+'. '+confound
elif not cleaner_ok:
    ship=False; ship_reason='Overall OK but cleaner ds007262 does not clear 0.55. '+confound
else:
    ship=True; ship_reason='Clears bar incl. ds007262; still proxy/research only. '+confound
log(f'ship_candidate={ship} reason={ship_reason}')

torch.save({'state_dict': head.state_dict(), 'label_list': list(HEAD_A_ENG_LABELS),
            'in_dim': int(emb_fit.shape[-1]), 'encoder': 'REVE', 'frozen_backbone': True,
            'accelerator': gpu_name, 'device_used': str(device)},
           OUT / 'head_a_eng_linear.pt')
created = datetime.now(timezone.utc).isoformat()
metrics = {
    'created_utc': created, 'encoder': 'REVE', 'frozen_backbone': True,
    'chance_macro_f1': CHANCE, 'accelerator': gpu_name, 'device_used': str(device),
    'train_fit': train_m, 'val': val_m, 'test': test_m, 'test_by_source': by_source,
    'history': history, 'ship_candidate': ship, 'ship_reason': ship_reason,
    'train_pool': {'n': int(len(y_tr)), 'sources': det_tr['sources'], 'device_classes': det_tr['device_classes']},
    'val_pool': {'n': int(len(y_va)), 'sources': det_va['sources']},
    'test_pool': {'n': int(len(y_te)), 'sources': det_te['sources'], 'device_classes': det_te['device_classes']},
}
(OUT / 'metrics_summary.json').write_text(json.dumps(metrics, indent=2))
(OUT / 'step_summary.json').write_text(json.dumps({
    'test_macro_f1': test_m['macro_f1'], 'delta_vs_chance': test_m['macro_f1']-CHANCE,
    'ship_candidate': ship, 'ship_reason': ship_reason, 'accelerator': gpu_name, 'device_used': str(device),
}, indent=2))
(OUT / 'run.log').write_text('\n'.join(LOG)+'\n')
log(f'wrote {OUT}')
print(json.dumps({'test_macro_f1': test_m['macro_f1'], 'delta_vs_chance': test_m['macro_f1']-CHANCE, 'ship_candidate': ship, 'device_used': str(device)}, indent=2))
